In [1]:
import pandas as pd

# Load data
df = pd.read_csv("../../data/detailed_games_sample.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (1982, 14)

Columns:
['player_username', 'player_sex', 'game_id', 'opponent_rating', 'opponent_id', 'is_white', 'won', 'status', 'total_moves', 'acpl', 'eco', 'opening_name', 'timestamp', 'moves']


In [4]:
#Check resignation signal
print("\nGame status counts:")
print(df["status"].value_counts())

resign_rate = (df["status"] == "resign").mean()
print(f"\nResignation rate: {resign_rate:.2%}")
#Inspect resignation timing (core proxy)
resign_games = df[df["status"] == "resign"]
print("\nResignation games count:", len(resign_games))
print("\nTotal moves (resignation games) summary:")
print(resign_games["total_moves"].describe())


Game status counts:
status
resign                       984
outoftime                    433
mate                         374
draw                         143
timeout                       21
noStart                       16
stalemate                     10
insufficientMaterialClaim      1
Name: count, dtype: int64

Resignation rate: 49.65%

Resignation games count: 984

Total moves (resignation games) summary:
count    984.000000
mean      33.794715
std       14.188261
min        0.000000
25%       24.000000
50%       32.000000
75%       42.000000
max      115.000000
Name: total_moves, dtype: float64


In [5]:
#Check per-player sample sizes
resign_counts = resign_games.groupby("player_username").size()

print("\nPlayers with resignation games:")
print(resign_counts.describe())

print("\nTop players by resignation count:")
print(resign_counts.sort_values(ascending=False).head(10))


Players with resignation games:
count    100.000000
mean       9.840000
std        2.997373
min        2.000000
25%        8.000000
50%       10.000000
75%       12.000000
max       17.000000
dtype: float64

Top players by resignation count:
player_username
Roman2001           17
gravereaper         16
MrSuccess           15
Lukianoo            15
theKittycat         14
itore               14
laksshana           14
juancruzariasTDF    14
Tenamu              14
athemm              14
dtype: int64


In [6]:
#ACPL sanity check (for Tilt)
print("\nACPL summary (all games):")
print(df["acpl"].describe())

print("\nMissing ACPL values:", df["acpl"].isna().sum())

#Time ordering check (Tilt prerequisite)
print("\nTimestamp summary:")
print(df["timestamp"].describe())


ACPL summary (all games):
count    1854.000000
mean       38.140777
std        22.581820
min         2.000000
25%        22.000000
50%        34.000000
75%        49.000000
max       196.000000
Name: acpl, dtype: float64

Missing ACPL values: 128

Timestamp summary:
count    1.982000e+03
mean     1.736366e+12
std      4.176986e+10
min      1.521850e+12
25%      1.730881e+12
50%      1.754229e+12
75%      1.764244e+12
max      1.765816e+12
Name: timestamp, dtype: float64


In [7]:
# Check per-player ordering
sample_player = df["player_username"].iloc[0]
sample_games = df[df["player_username"] == sample_player].sort_values("timestamp")
print(f"\nSample games for player: {sample_player}")
print(sample_games[["timestamp", "status", "won", "total_moves"]].head())


Sample games for player: elsas
        timestamp     status    won  total_moves
19  1765313179051     resign  False           59
18  1765376666374     resign  False           60
17  1765401827752       draw  False           62
16  1765402219741  outoftime   True           48
15  1765402607898  outoftime   True           48


In [8]:


# Keep only resignation losses
resign_df = df[
    (df["status"] == "resign") &
    (df["won"] == False) &
    (df["total_moves"] > 0)   # remove pathological cases
].copy()

print("Valid resignation games:", len(resign_df))


Valid resignation games: 363


In [10]:
# Compute Resignation Threshold per player
rt_player = (
    resign_df
    .groupby("player_username")["total_moves"]
    .median()
    .reset_index()
    .rename(columns={"total_moves": "resignation_threshold"})
)

# Add sample size (important for interpretation)
resign_counts = resign_df.groupby("player_username").size().reset_index(name="n_resignations")

rt_player = rt_player.merge(resign_counts, on="player_username")

print(rt_player.head())

# Save output
output_path = "resignation_threshold_v1.csv"
rt_player.to_csv(output_path, index=False)

print(f"Saved: {output_path}")



  player_username  resignation_threshold  n_resignations
0        AGmedina                   28.0               2
1        Alindsay                   32.0               5
2      Anchkaaaaa                   45.0               3
3        ArmQueen                   39.0               7
4         Banzeus                   33.5               4
Saved: resignation_threshold_v1.csv


In [11]:
print("\nResignation Threshold summary:")
print(rt_player["resignation_threshold"].describe())

print("\nPlayers with >= 5 resignation games:")
print((rt_player["n_resignations"] >= 5).sum())



Resignation Threshold summary:
count    90.000000
mean     34.844444
std       8.425031
min      15.000000
25%      29.625000
50%      34.000000
75%      39.500000
max      63.000000
Name: resignation_threshold, dtype: float64

Players with >= 5 resignation games:
31
